<a href="https://colab.research.google.com/github/1pawn0/time-series-forecasting-lab/blob/main/LightGBM_time_series_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU lightgbm polars scikit-learn

In [ ]:
import os
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import polars as pl
import lightgbm as lgb


In [ ]:
from urllib.request import urlretrieve
from pathlib import Path

csv_url: str = "https://www.cryptodatadownload.com/cdd/Gemini_BTCUSD_1h.csv"
data_dir: Path = Path("./data")
data_dir.mkdir(exist_ok=True, parents=True)
csv_file: Path = data_dir / "Gemini_BTCUSD_1h.csv"

if not csv_file.exists():
    urlretrieve(csv_url, csv_file)
    print(f"CSV file downloaded to {csv_file}")
else:
    print(f"CSV file already exists at {csv_file}")

import polars as pl

# Define the schema of df
df_schema: dict = {
    "date": pl.Datetime("ms"),
    "close": pl.Float64,
}
# Define which columns to load from the CSV
cols = list(df_schema.keys())
# Read the CSV file
df = pl.read_csv(csv_file, columns=cols, skip_lines=1, schema_overrides=df_schema).sort("date").rename({"close": "price"})
# Fill the date gaps inside `df`
full_date_range = pl.datetime_range(
    start=df["date"][0],
    end=df["date"][-1],
    interval="1h",
    time_unit="ms",
    eager=True,
).to_frame(name="date")
df = full_date_range.join(df, on="date", how="left").interpolate().sort("date")
# Filter the dataset to only include data after a specific start date
df = df.filter(pl.col("date") > datetime(2016, 10, 30)).sort("date")
price_pct_changes_df = df.with_columns(pl.col("price").pct_change().shift(-1).alias("price_pct_change")).sort("date")[:-1]
price_pct_changes_df


In [ ]:
prices = price_pct_changes_df.with_columns(
    [
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.day().alias("day"),
        pl.col("date").dt.hour().alias("hour"),
        pl.col("date").dt.weekday().alias("weekday"),
        pl.col("date").dt.week().alias("week"),
        pl.col("date").dt.quarter().alias("quarter"),
        pl.col("date").dt.ordinal_day().alias("ordinal_day"),
    ]
).sort("date")


In [ ]:
hours: int = 24
lag_expressions: list[pl.Expr] = [
    pl.col("price_pct_change").shift(i).alias(f"pct_change_lag_{i}h") for i in range(1, hours + 1)
]
rolling_expressions: list[pl.Expr] = [
    pl.col("price_pct_change").shift().rolling_mean(hours).alias(f"rolling_mean_{hours}h"),
    pl.col("price_pct_change").shift().rolling_std(hours).alias(f"rolling_std_{hours}h"),
    pl.col("price_pct_change").shift().rolling_var(hours).alias(f"rolling_var_{hours}h"),
    pl.col("price_pct_change").shift().rolling_skew(hours).alias(f"rolling_skew_{hours}h"),
    pl.col("price_pct_change").shift().rolling_kurtosis(hours).alias(f"rolling_kurtosis_{hours}h"),
    pl.col("price_pct_change").shift().ewm_mean_by(by="date", half_life="24h").alias(f"ewm_mean_{hours}h"),
]
expr_list = lag_expressions + rolling_expressions

lagged_df = prices.with_columns(expr_list).drop_nulls().drop("price")


In [ ]:
cyclical_periods = {
    "month": 12,
    "day": 31,
    "hour": 24,
    "weekday": 7,
    "week": 53,
    "quarter": 4,
    "ordinal_day": 366,
}

cols_to_process = [col for col in cyclical_periods if col in lagged_df.columns]

expressions = []
for col_name in cols_to_process:
    max_val = cyclical_periods[col_name]
    col_expr = pl.col(col_name)

    expressions.append(
        (col_expr * (2 * np.pi / max_val)).sin().alias(f"{col_name}_sine")
    )
    expressions.append(
        (col_expr * (2 * np.pi / max_val)).cos().alias(f"{col_name}_cosine")
    )

processed_df = lagged_df.with_columns(expressions).drop(cols_to_process)

processed_df


In [ ]:
# Split the test set
split_point = processed_df.height - (hours**2)
train_df = processed_df.slice(0, split_point)
test_df = processed_df.slice(split_point, processed_df.height)
X_test, y_test = (
    test_df.drop("price_pct_change").drop(["date", "year"]),
    test_df["price_pct_change"],
)
X_test.shape, y_test.shape, test_df.shape, train_df.shape, processed_df.shape

# Split the train and validation sets
split_point_val = int(0.95 * train_df.height)
X, y = train_df.drop("price_pct_change"), train_df["price_pct_change"]
X_train = X.slice(0, split_point_val).drop(["date", "year"])
y_train = y.slice(0, split_point_val)
X_val = X.slice(split_point_val, train_df.height).drop(["date", "year"])
y_val = y.slice(split_point_val, train_df.height)

print(
    f" X_train: {X_train.shape}\n y_train: {y_train.shape}\n X_val: {X_val.shape}\n y_val: {y_val.shape}\n X_test: {X_test.shape}\n y_test: {y_test.shape}"
)


In [ ]:
import lightgbm as lgb

lgbm_params = {
    "task": "train",
    "objective": "l2",
    "metric": "",
    "boosting": "dart",
    "data_sample_strategy": "bagging",
    "num_iterations": 50,
    "learning_rate": 0.1,
    "num_leaves": 16_383,
    "tree_learner": "voting",
    "seed": 42,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "bagging_fraction": 0.6,
    "bagging_freq": 2,
    "early_stopping_round": 10,
    "lambda_l1": 0,
    "lambda_l2": 0.5,
    "drop_rate": 0.2,
    "max_drop": -1,
    "skip_drop": 0.1,
    "uniform_drop": True,
    "top_k": 50,
    "refit_decay_rate": 0.9,
    "path_smooth": 1.0,
    "verbosity": 3,
    "use_quantized_grad": False,
    "linear_tree": False,
    "max_bin": 255,
    "min_data_in_bin": 7,
    "use_missing": False,
    "feature_pre_filter": True,
    "two_round": False,
    "save_binary": True,
}
train_ds = lgb.Dataset(
    data=X_train.to_numpy(),
    label=y_train.to_numpy(),
    params=lgbm_params,
    free_raw_data=False,
)
val_ds = lgb.Dataset(
    data=X_val.to_numpy(),
    label=y_val.to_numpy(),
    reference=train_ds,
    params=lgbm_params,
    free_raw_data=False,
)

booster = lgb.Booster(params=lgbm_params, train_set=train_ds, model_file="lgbm_model")


In [ ]:
trained_booster = lgb.train(
    params=lgbm_params,
    train_set=train_ds,
    num_boost_round=50,
    valid_sets=[train_ds, val_ds],
    valid_names=["train", "validation"],
    init_model=booster,
    keep_training_booster=True,
)
